In [0]:

from pyspark.sql.functions import current_timestamp, col
import os

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

source_path = "/Volumes/team_8/quantum_force/quantum_force"

catalog_name = "team_8"
schema_name = "quantum_force"

# --------------------------------------------------
# LIST JSON FILES
# --------------------------------------------------

files = dbutils.fs.ls(source_path)

json_files = [
    file for file in files
    if file.name.lower().endswith(".json")
]

print(f"Total JSON files found: {len(json_files)}")

# --------------------------------------------------
# LOAD JSON FILES INTO BRONZE TABLES
# --------------------------------------------------

for file in json_files:

    table_name = os.path.splitext(file.name)[0].lower()
    target_table = f"{catalog_name}.{schema_name}.{table_name}"

    print(f"\nProcessing file: {file.name}")
    print(f"Target table: {target_table}")

    try:

        # Read JSON using PySpark
        df = (
            spark.read
            .option("multiLine", "true")
            .json(file.path)
        )

        # Add audit columns
        df = df.withColumns({
            "_source_file": col("_metadata.file_path"),
            "_bronze_load_timestamp": current_timestamp()
        })

        # Check whether Bronze table already exists
        table_exists = spark.catalog.tableExists(target_table)

        if table_exists:

            print(f"Table exists: {target_table}")
            print("Overwriting existing Bronze table...")

        else:

            print(f"Table does not exist: {target_table}")
            print("Creating new Bronze table...")

        # Create or overwrite Bronze table
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )

        # Validate record count
        record_count = spark.table(target_table).count()

        print(f"SUCCESS: {target_table}")
        print(f"Records loaded: {record_count}")

    except Exception as e:

        print(f"ERROR loading {file.name}")
        print(str(e).split("JVM stacktrace")[0])

# --------------------------------------------------
# VALIDATE CREATED TABLES
# --------------------------------------------------

tables = spark.catalog.listTables(f"{catalog_name}.{schema_name}")

print("\nBRONZE TABLES CREATED:")

for table in tables:
    print(table.name)

print("\nBRONZE INGESTION COMPLETED!")


Total JSON files found: 18

Processing file: data_quality_audit.json
Target table: team_8.quantum_force.data_quality_audit
Table exists: team_8.quantum_force.data_quality_audit
Overwriting existing Bronze table...
SUCCESS: team_8.quantum_force.data_quality_audit
Records loaded: 1

Processing file: developers.json
Target table: team_8.quantum_force.developers
Table exists: team_8.quantum_force.developers
Overwriting existing Bronze table...
SUCCESS: team_8.quantum_force.developers
Records loaded: 1190

Processing file: dim_calendar.json
Target table: team_8.quantum_force.dim_calendar
Table exists: team_8.quantum_force.dim_calendar
Overwriting existing Bronze table...
SUCCESS: team_8.quantum_force.dim_calendar
Records loaded: 5960

Processing file: dim_time.json
Target table: team_8.quantum_force.dim_time
Table exists: team_8.quantum_force.dim_time
Overwriting existing Bronze table...
SUCCESS: team_8.quantum_force.dim_time
Records loaded: 86400

Processing file: epics.json
Target table: 

In [0]:
# # --------------------------------------------------
# # INCREMENTAL LOAD: ACTIVE SPRINT (SPRINT-0401, PRJ-001 Kafka Core)
# # Merges 3 JSON files into EXISTING Bronze tables using Delta MERGE
# # (re-runnable: no duplicate issue / sprint / bridge rows).
# #
# # Run AFTER the full Bronze ingestion cell above.
# # Keep these files in a SUB-FOLDER (not the root source_path), otherwise the
# # ingestion cell above would create extra tables named after them.
# # If the ingestion cell above is re-run (overwrite), re-run this cell too.
# # --------------------------------------------------

# from pyspark.sql import functions as F
# from pyspark.sql.types import ArrayType, StructType

# SOURCE_DIR = "/Volumes/team_8/quantum_force/quantum_force"   

# catalog_name = "team_8"
# schema_name = "quantum_force"

# SOURCE_FILES = [
#     "active_sprint.json",
#     "active_sprint_issues.json",
#     "active_sprint_record.json",
# ]

# # Business keys per existing Bronze table (match the Gold primary keys)
# MERGE_KEYS = {
#     "sprints": ["sprint_id"],
#     "issues_enriched": ["issue_id"],
#     "project_sprint_bridge": ["project_id", "sprint_id"],
# }
# LOAD_ORDER = ["sprints", "issues_enriched", "project_sprint_bridge"]
# AUDIT_COLS = ["_source_file", "_bronze_load_timestamp"]


# def read_json(path):
#     df = spark.read.option("multiLine", "true").json(path)
#     if "_corrupt_record" in df.columns:
#         raise ValueError(f"Malformed JSON in {path}")
#     # Unwrap {"records": [ {...}, ... ]} style wrappers
#     if len(df.columns) == 1:
#         dtype = df.schema[0].dataType
#         if isinstance(dtype, ArrayType) and isinstance(dtype.elementType, StructType):
#             df = df.select(F.explode(df.columns[0]).alias("_r")).select("_r.*")
#     return df


# def detect_target(df):
#     # Decide the target table from the file's columns, not its name
#     cols = set(df.columns)
#     if "issue_id" in cols:
#         return "issues_enriched"
#     if {"sprint_id", "sprint_name"} <= cols:
#         return "sprints"
#     if {"project_id", "sprint_id"} <= cols:
#         return "project_sprint_bridge"
#     return None


# def align_to_target(df, target_table, source_file):
#     # Match the existing table schema: same columns, same types, audit columns filled
#     target_schema = spark.table(target_table).schema
#     target_cols = [f.name for f in target_schema]

#     extra = [c for c in df.columns if c not in target_cols]
#     missing = [c for c in target_cols if c not in df.columns and c not in AUDIT_COLS]
#     if extra:
#         print(f"  WARNING: columns not in {target_table}, ignored: {extra}")
#     if missing:
#         print(f"  INFO: columns missing in file, loaded as NULL: {missing}")

#     exprs = []
#     for f in target_schema:
#         if f.name == "_source_file":
#             e = F.lit(source_file)
#         elif f.name == "_bronze_load_timestamp":
#             e = F.current_timestamp()
#         elif f.name in df.columns:
#             e = F.col(f"`{f.name}`").cast(f.dataType)
#         else:
#             e = F.lit(None).cast(f.dataType)
#         exprs.append(e.alias(f.name))
#     return df.select(*exprs)


# # --------------------------------------------------
# # READ AND MAP FILES
# # --------------------------------------------------

# def is_manifest(df):
#     # active_sprint.json: sprint + project_id + assigned issue list (not a table record)
#     return {"sprint", "project_id", "assigned_issue_ids"} <= set(df.columns)


# staged = {}
# manifest = None
# for file_name in SOURCE_FILES:
#     path = f"{SOURCE_DIR.rstrip('/')}/{file_name}"
#     df = read_json(path)
#     if is_manifest(df):
#         manifest = (file_name, path, df)
#         print(f"{file_name:<30} -> sprint assignment manifest (bridge + validation)")
#         continue
#     target = detect_target(df)
#     if target is None:
#         raise ValueError(
#             f"Cannot map {file_name} to a Bronze table. Columns: {df.columns}"
#         )
#     if target in staged:
#         raise ValueError(
#             f"{file_name} and {staged[target][0]} both map to {target}"
#         )
#     staged[target] = (file_name, path, df)
#     print(f"{file_name:<30} -> {target:<22} rows={df.count()}")

# # Derive bridge (and sprint record if not supplied by another file) from the manifest
# if manifest:
#     m_file, m_path, m_df = manifest
#     sprint_type = m_df.schema["sprint"].dataType
#     if isinstance(sprint_type, StructType):
#         sprint_fields = sprint_type.fieldNames()
#         if "sprint_id" not in sprint_fields:
#             raise ValueError(f"{m_file}: 'sprint' has no sprint_id. Fields: {sprint_fields}")
#         sprint_id_col = F.col("sprint.sprint_id")
#     else:
#         sprint_fields = []
#         sprint_id_col = F.col("sprint")

#     if "project_sprint_bridge" not in staged:
#         bridge_df = m_df.select(
#             F.col("project_id"), sprint_id_col.alias("sprint_id")
#         ).distinct()
#         staged["project_sprint_bridge"] = (m_file, m_path, bridge_df)
#         print(f"{m_file:<30} -> project_sprint_bridge  rows={bridge_df.count()}")

#     if "sprints" not in staged and "sprint_name" in sprint_fields:
#         sprint_df = (
#             m_df.select("sprint.*") if "project_id" in sprint_fields
#             else m_df.select("sprint.*", "project_id")
#         )
#         staged["sprints"] = (m_file, m_path, sprint_df)
#         print(f"{m_file:<30} -> sprints                rows={sprint_df.count()}")

# # --------------------------------------------------
# # MERGE INTO EXISTING BRONZE TABLES
# # --------------------------------------------------

# merge_results = []
# for table in LOAD_ORDER:
#     if table not in staged:
#         print(f"\nNo file for {table}, skipped.")
#         continue

#     file_name, path, df = staged[table]
#     target_table = f"{catalog_name}.{schema_name}.{table}"
#     keys = MERGE_KEYS[table]

#     if not spark.catalog.tableExists(target_table):
#         raise ValueError(f"{target_table} does not exist. Run the Bronze ingestion first.")

#     print(f"\nMerging {file_name} into {target_table} on {keys}")
#     aligned = align_to_target(df, target_table, path)

#     null_keys = aligned.filter(
#         " OR ".join(f"`{k}` IS NULL" for k in keys)
#     ).count()
#     if null_keys:
#         raise ValueError(f"{file_name}: {null_keys} rows have NULL merge keys {keys}")

#     src_rows = aligned.count()
#     aligned = aligned.dropDuplicates(keys)
#     if aligned.count() != src_rows:
#         print(f"  WARNING: {src_rows - aligned.count()} duplicate key rows dropped from file")

#     view = f"_src_{table}"
#     aligned.createOrReplaceTempView(view)

#     cols = aligned.columns
#     on_clause = " AND ".join(f"t.`{k}` = s.`{k}`" for k in keys)
#     set_clause = ", ".join(f"t.`{c}` = s.`{c}`" for c in cols if c not in keys)
#     insert_cols = ", ".join(f"`{c}`" for c in cols)
#     insert_vals = ", ".join(f"s.`{c}`" for c in cols)

#     result = spark.sql(f"""
#         MERGE INTO {target_table} t
#         USING {view} s
#         ON {on_clause}
#         WHEN MATCHED THEN UPDATE SET {set_clause}
#         WHEN NOT MATCHED THEN INSERT ({insert_cols}) VALUES ({insert_vals})
#     """).collect()[0].asDict()

#     # Verify every key from the file is now present exactly once in the table
#     in_target = (
#         spark.table(target_table)
#         .join(aligned.select(*keys), keys, "left_semi")
#         .groupBy(*keys).count()
#     )
#     present = in_target.count()
#     dup_in_target = in_target.filter("count > 1").count()
#     expected = aligned.count()

#     print(f"  inserted={result.get('num_inserted_rows')} "
#           f"updated={result.get('num_updated_rows')} "
#           f"keys present={present}/{expected} duplicates={dup_in_target}")

#     if present != expected or dup_in_target:
#         raise ValueError(f"Post-merge check failed for {target_table}")

#     merge_results.append((
#         table, file_name, expected,
#         result.get("num_inserted_rows"), result.get("num_updated_rows"),
#         spark.table(target_table).count()
#     ))

# display(spark.createDataFrame(
#     merge_results,
#     "table STRING, source_file STRING, file_rows LONG, inserted LONG, updated LONG, table_total_rows LONG"
# ))

# # --------------------------------------------------
# # ACTIVE SPRINT SANITY CHECKS (expected: 24 issues, 8 teams x 3,
# # 12 Open / 12 In Progress, story points 1-3)
# # --------------------------------------------------

# if "issues_enriched" in staged:
#     new_issues = (
#         spark.table(f"{catalog_name}.{schema_name}.issues_enriched")
#         .join(
#             align_to_target(staged["issues_enriched"][2],
#                             f"{catalog_name}.{schema_name}.issues_enriched", "")
#             .select("issue_id"),
#             "issue_id", "left_semi"
#         )
#     )
#     print(f"\nActive sprint issues in Bronze: {new_issues.count()}")
#     display(new_issues.groupBy("sprint_id", "project_id").count())
#     display(new_issues.groupBy("team_id").count().orderBy("team_id"))
#     display(new_issues.groupBy("status").count())
#     display(new_issues.agg(
#         F.min("story_points").alias("min_story_points"),
#         F.max("story_points").alias("max_story_points"),
#         F.sum("story_points").alias("total_story_points")
#     ))

# # Manifest check: every assigned issue id must now exist in issues_enriched
# if manifest:
#     m_ids = (
#         m_df.select(F.explode("assigned_issue_ids").alias("issue_id"))
#         .select(F.col("issue_id").cast("string").alias("issue_id"))
#         .distinct()
#     )
#     expected_ids = m_ids.count()
#     declared = m_df.select(F.sum("assignment_count")).first()[0]
#     found_ids = (
#         spark.table(f"{catalog_name}.{schema_name}.issues_enriched")
#         .select(F.col("issue_id").cast("string").alias("issue_id"))
#         .join(m_ids, "issue_id", "left_semi")
#         .distinct().count()
#     )
#     print(f"\nManifest: assignment_count={declared}, assigned ids={expected_ids}, "
#           f"found in issues_enriched={found_ids}")
#     if found_ids != expected_ids or (declared is not None and declared != expected_ids):
#         raise ValueError("Manifest check failed: assigned issues missing or count mismatch")

# print("\nACTIVE SPRINT MERGE COMPLETED. Re-run the Bronze-to-Gold notebook to refresh Gold.")

active_sprint.json             -> sprint assignment manifest (bridge + validation)
active_sprint_issues.json      -> issues_enriched        rows=24
active_sprint_record.json      -> sprints                rows=1
active_sprint.json             -> project_sprint_bridge  rows=1

Merging active_sprint_record.json into team_8.quantum_force.sprints on ['sprint_id']
  inserted=1 updated=0 keys present=1/1 duplicates=0

Merging active_sprint_issues.json into team_8.quantum_force.issues_enriched on ['issue_id']
  inserted=24 updated=0 keys present=24/24 duplicates=0

Merging active_sprint.json into team_8.quantum_force.project_sprint_bridge on ['project_id', 'sprint_id']
  INFO: columns missing in file, loaded as NULL: ['data_type', 'is_primary_project', 'issue_count']
  inserted=1 updated=0 keys present=1/1 duplicates=0


table,source_file,file_rows,inserted,updated,table_total_rows
sprints,active_sprint_record.json,1,1,0,401
issues_enriched,active_sprint_issues.json,24,24,0,20201
project_sprint_bridge,active_sprint.json,1,1,0,1970



Active sprint issues in Bronze: 24


sprint_id,project_id,count
SPRINT-0401,PRJ-001,24


team_id,count
TEAM-BA,3
TEAM-BACKEND,3
TEAM-DEVOPS,3
TEAM-FRONTEND,3
TEAM-PLATFORM,3
TEAM-PO,3
TEAM-QA,3
TEAM-SCRUM,3


status,count
Open,12
In Progress,12


min_story_points,max_story_points,total_story_points
1,3,48



Manifest: assignment_count=24, assigned ids=24, found in issues_enriched=24

ACTIVE SPRINT MERGE COMPLETED. Re-run the Bronze-to-Gold notebook to refresh Gold.
